<h2>Joining Sequence/metadata Information with Interaction datset</h2>
<h6>We keep only interaction with avaiable sequences in both Interacting RNAs</h6>

In [5]:
import pandas as pd

# load interactions
df_filt = pd.read_pickle("data/interactions_processed_noSeq.pkl")
rna_entities_ncRNA = pd.read_pickle("data/nc_rna_entities_processed.pkl")
rna_entities_mRNA = pd.read_pickle("data/m_rna_entities_processed.pkl") 

In [6]:
print(len(df_filt))
print(len(rna_entities_ncRNA))
print(len(rna_entities_mRNA))

9429404
40911
197490


In [10]:
print(rna_entities_ncRNA.keys())
print(rna_entities_mRNA.keys())
print(df_filt.keys())

Index(['Category', 'ID_buckets', 'RNA_IDs', 'Raw_ID_key', 'Species', 'Symbol',
       'sequences', 'typecheck', 'GO', 'Rfam', 'GO_raw', 'GO_BP_count',
       'GO_MF_count', 'GO_CC_count', 'mygene_raw'],
      dtype='object')
Index(['Symbol', 'Category', 'Species', 'Raw_ID_key', 'RNA_IDs', 'ID_buckets',
       'refseq', 'ensembl_ids', 'ensembl_seq', 'refseq_seq', 'sequences',
       'GO_raw', 'GO_BP_count', 'GO_MF_count', 'GO_CC_count', 'mygene_raw'],
      dtype='object')
Index(['RNAInterID', 'Interactor1.Symbol', 'Category1', 'Species1',
       'Interactor2.Symbol', 'Category2', 'Species2', 'Raw_ID1', 'Raw_ID2',
       'score', 'strong', 'weak', 'predict', 'Raw_ID1_list', 'Raw_ID2_list',
       'Species1_TaxID', 'Species2_TaxID', 'Category1_std', 'Category2_std',
       'Raw_ID1_key', 'Raw_ID2_key'],
      dtype='object')


In [11]:
import pandas as pd

# ==========================================================
# 1. Normalize Raw_ID keys in interaction dataframe
# ==========================================================
def normalize_raw_ids(x):
    if isinstance(x, list):
        return tuple(x)
    return x

df_filt["Raw_ID1_key"] = df_filt["Raw_ID1_list"].apply(normalize_raw_ids)
df_filt["Raw_ID2_key"] = df_filt["Raw_ID2_list"].apply(normalize_raw_ids)

print("\nSample normalized keys")
print(df_filt[["Raw_ID1_key","Raw_ID2_key"]].head())


# ==========================================================
# 2. Build interaction keys
# ==========================================================
df_filt["key1"] = list(zip(
    df_filt["Interactor1.Symbol"],
    df_filt["Category1"],
    df_filt["Species1_TaxID"],
    df_filt["Raw_ID1_key"]
))

df_filt["key2"] = list(zip(
    df_filt["Interactor2.Symbol"],
    df_filt["Category2"],
    df_filt["Species2_TaxID"],
    df_filt["Raw_ID2_key"]
))

print("\nSample interaction keys")
print(df_filt[["key1","key2"]].head())


# ==========================================================
# 3. Prepare entity tables
# ==========================================================
def prepare_entities(df, name):

    df = df.copy()

    df["Raw_key"] = df["RNA_IDs"].apply(normalize_raw_ids)

    df["entity_key"] = list(zip(
        df["Symbol"],
        df["Category"],
        df["Species"],
        df["Raw_key"]
    ))

    print(f"\n{name} entities:", len(df))
    print(df[["Symbol","Category","Species","Raw_key"]].head())

    return df   # KEEP ALL COLUMNS


entities_nc = prepare_entities(rna_entities_ncRNA, "ncRNA")
entities_mr = prepare_entities(rna_entities_mRNA, "mRNA")

entities_all = pd.concat([entities_nc, entities_mr], ignore_index=True)

print("\nTotal entity entries:", len(entities_all))


# ==========================================================
# 4. Remove duplicate entity keys
# ==========================================================
entities_all = entities_all.drop_duplicates("entity_key")

print("Unique entity keys:", len(entities_all))


# ==========================================================
# 5. Prepare prefixed entity tables for merge
# ==========================================================
entities1 = entities_all.add_prefix("ent1_")
entities1 = entities1.rename(columns={"ent1_entity_key":"key1"})

entities2 = entities_all.add_prefix("ent2_")
entities2 = entities2.rename(columns={"ent2_entity_key":"key2"})


# ==========================================================
# 6. Merge entity metadata onto interactions
# ==========================================================
df_merged = df_filt.merge(
    entities1,
    on="key1",
    how="left"
)

df_merged = df_merged.merge(
    entities2,
    on="key2",
    how="left"
)


print("\nMerged sample")
print(df_merged[[
    "key1","ent1_sequences",
    "key2","ent2_sequences"
]].head())


# ==========================================================
# 7. Diagnostics
# ==========================================================
missing1 = df_merged["ent1_sequences"].isna().sum()
missing2 = df_merged["ent2_sequences"].isna().sum()

print("\nMissing sequences:")
print("Interactor1:", missing1)
print("Interactor2:", missing2)

print("\nExample missing keys (Interactor1)")
print(df_merged[df_merged["ent1_sequences"].isna()]["key1"].head())

print("\nExample missing keys (Interactor2)")
print(df_merged[df_merged["ent2_sequences"].isna()]["key2"].head())


# ==========================================================
# 8. mRNA statistics (no filtering of rows)
# ==========================================================
def is_mrna(key):
    return str(key[1]).lower() in ["mrna","protein_coding"]

df_merged["interactor1_mRNA"] = df_merged["key1"].apply(is_mrna)
df_merged["interactor2_mRNA"] = df_merged["key2"].apply(is_mrna)

print("\nInteractions involving ≥1 mRNA:",
      (df_merged["interactor1_mRNA"] | df_merged["interactor2_mRNA"]).sum())


print("\nFinal interaction dataframe shape:", df_merged.shape)


Sample normalized keys
                        Raw_ID1_key                    Raw_ID2_key
3638  (Ensembl:ENSMUSG00000065862,)  (Ensembl:ENSMUSG00000070228,)
3639  (Ensembl:ENSMUSG00000065862,)  (Ensembl:ENSMUSG00000075763,)
3640  (Ensembl:ENSMUSG00000065862,)  (Ensembl:ENSMUSG00000064881,)
3641  (Ensembl:ENSMUSG00000065282,)  (Ensembl:ENSMUSG00000064430,)
3642  (Ensembl:ENSMUSG00000065282,)  (Ensembl:ENSMUSG00000064881,)

Sample interaction keys
                                                   key1  \
3638  (Gm24029, snoRNA, 10090, (Ensembl:ENSMUSG00000...   
3639  (Gm24029, snoRNA, 10090, (Ensembl:ENSMUSG00000...   
3640  (Gm24029, snoRNA, 10090, (Ensembl:ENSMUSG00000...   
3641  (Gm24455, snoRNA, 10090, (Ensembl:ENSMUSG00000...   
3642  (Gm24455, snoRNA, 10090, (Ensembl:ENSMUSG00000...   

                                                   key2  
3638  (U5, snRNA, 10090, (Ensembl:ENSMUSG00000070228,))  
3639  (U5, snRNA, 10090, (Ensembl:ENSMUSG00000075763,))  
3640  (U5, snRNA, 10

In [12]:
# ==========================================================
# 8. Filter valid interactions (require sequences on both)
# ==========================================================
has_seq1 = df_merged["ent1_sequences"].apply(
    lambda x: isinstance(x, list) and len(x) > 0
)

has_seq2 = df_merged["ent2_sequences"].apply(
    lambda x: isinstance(x, list) and len(x) > 0
)

df_filtered = df_merged[has_seq1 & has_seq2].copy()

print("\nInteractions before filtering:", len(df_merged))
print("Interactions after filtering:", len(df_filtered))


Interactions before filtering: 9429404
Interactions after filtering: 7449338


In [13]:
# ==========================================================
# 9. mRNA statistics
# ==========================================================
def is_mrna(key):
    return str(key[1]).lower() in ["mrna", "protein_coding"]

df_filtered["interactor1_mRNA"] = df_filtered["key1"].apply(is_mrna)
df_filtered["interactor2_mRNA"] = df_filtered["key2"].apply(is_mrna)

print(
    "\nInteractions involving ≥1 mRNA:",
    (df_filtered["interactor1_mRNA"] | df_filtered["interactor2_mRNA"]).sum()
)


Interactions involving ≥1 mRNA: 7329973


In [15]:
df_filtered.keys()

Index(['RNAInterID', 'Interactor1.Symbol', 'Category1', 'Species1',
       'Interactor2.Symbol', 'Category2', 'Species2', 'Raw_ID1', 'Raw_ID2',
       'score', 'strong', 'weak', 'predict', 'Raw_ID1_list', 'Raw_ID2_list',
       'Species1_TaxID', 'Species2_TaxID', 'Category1_std', 'Category2_std',
       'Raw_ID1_key', 'Raw_ID2_key', 'key1', 'key2', 'ent1_Category',
       'ent1_ID_buckets', 'ent1_RNA_IDs', 'ent1_Raw_ID_key', 'ent1_Species',
       'ent1_Symbol', 'ent1_sequences', 'ent1_typecheck', 'ent1_GO',
       'ent1_Rfam', 'ent1_GO_raw', 'ent1_GO_BP_count', 'ent1_GO_MF_count',
       'ent1_GO_CC_count', 'ent1_mygene_raw', 'ent1_Raw_key', 'ent1_refseq',
       'ent1_ensembl_ids', 'ent1_ensembl_seq', 'ent1_refseq_seq',
       'ent2_Category', 'ent2_ID_buckets', 'ent2_RNA_IDs', 'ent2_Raw_ID_key',
       'ent2_Species', 'ent2_Symbol', 'ent2_sequences', 'ent2_typecheck',
       'ent2_GO', 'ent2_Rfam', 'ent2_GO_raw', 'ent2_GO_BP_count',
       'ent2_GO_MF_count', 'ent2_GO_CC_count', 'en

In [32]:
# redoundant columns that should be dropped.
# ==========================================================
#  Drop redundant columns
# ==========================================================

cols_to_drop = [
    'Category1_std', 'Category2_std',
    'ent1_Category', 'ent2_Category',
    'ent1_RNA_IDs', 'ent2_RNA_IDs',
    'ent1_Raw_ID_key', 'ent2_Raw_ID_key',
    'ent1_Species', 'ent2_Species',
    'ent1_Symbol', 'ent2_Symbol',
    'ent1_GO_MF_count', 'ent2_GO_MF_count',
    'ent1_GO_CC_count', 'ent2_GO_CC_count',
    'ent1_GO_BP_count', 'ent2_GO_BP_count',
    'ent1_refseq_seq', 'ent2_refseq_seq',
    'ent1_ensembl_seq', 'ent2_ensembl_seq',
    'Raw_ID1_list', 'Raw_ID2_list',
    'Raw_ID1_key', 'Raw_ID2_key',
    'ent1_refseq', 'ent2_refseq',
    'ent1_ensembl_ids', 'ent2_ensembl_ids',
    'interactor1_mRNA', 'interactor2_mRNA'
]

df_filtered = df_filtered.drop(
    columns=[c for c in cols_to_drop if c in df_filtered.columns]
)

print("\nFinal dataframe shape:", df_filtered.shape)
print("Remaining columns:")
print(df_filtered.columns)


Final dataframe shape: (7449338, 33)
Remaining columns:
Index(['RNAInterID', 'Interactor1.Symbol', 'Category1', 'Species1',
       'Interactor2.Symbol', 'Category2', 'Species2', 'Raw_ID1', 'Raw_ID2',
       'score', 'strong', 'weak', 'predict', 'Species1_TaxID',
       'Species2_TaxID', 'key1', 'key2', 'ent1_ID_buckets', 'ent1_sequences',
       'ent1_typecheck', 'ent1_GO', 'ent1_Rfam', 'ent1_GO_raw',
       'ent1_mygene_raw', 'ent1_Raw_key', 'ent2_ID_buckets', 'ent2_sequences',
       'ent2_typecheck', 'ent2_GO', 'ent2_Rfam', 'ent2_GO_raw',
       'ent2_mygene_raw', 'ent2_Raw_key'],
      dtype='object')


In [38]:
df_filtered = df_filtered.rename(columns={
    "ent1_GO": "ent1_GO_RNAcentral",
    "ent2_GO": "ent2_GO_RNAcentral"
})

In [41]:
df_filt_final = df_filtered
# Check for mRNA in interactor1 or interactor2
is_mrna1 = df_filt_final['Category1'].eq('mRNA')
is_mrna2 = df_filt_final['Category2'].eq('mRNA')

# Interactions involving at least one mRNA
interactions_with_mrna = df_filt_final[is_mrna1 | is_mrna2]
interactions_without_mrna = df_filt_final[~(is_mrna1 | is_mrna2)]

print(f"Total interactions: {len(df_filt_final)}")
print(f"Interactions involving at least one mRNA: {len(interactions_with_mrna)}")
print(f"Interactions with no mRNA: {len(interactions_without_mrna)}")

Total interactions: 7449338
Interactions involving at least one mRNA: 7329973
Interactions with no mRNA: 119365


In [42]:
df_filt_final.to_pickle("data/interactions_with_sequences_RNA_RNA_15-03-2026.pkl")

In [43]:
import pandas as pd

# Collect all entity_keys present in the filtered interactions
keys_retained = pd.concat([
    df_filt_final['key1'],
    df_filt_final['key2']
]).drop_duplicates()

print(f"Unique RNA entities retained: {len(keys_retained)}")


Unique RNA entities retained: 174248
